In [1]:
# --------------------------------------------------
# Project Root
# --------------------------------------------------

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\IITG\Projects\audio_factor_disentanglement_v2


In [2]:
from inspect import signature

from src.models.factorized.encoder_utils import SpectrogramToSequence
from src.models.factorized.base_blocks import ConformerStack
from src.models.factorized.base_blocks import TransformerStack
from src.models.factorized.base_blocks import DilatedTCN

print(signature(SpectrogramToSequence.__init__))
print(signature(ConformerStack.__init__))
print(signature(TransformerStack.__init__))
print(signature(DilatedTCN.__init__))

(self, input_freq_bins, hidden_dim)
(self, dim, num_layers, num_heads=8, dropout=0.1)
(self, dim, num_layers, num_heads=8, dropout=0.1)
(self, channels, layers=6, dropout=0.1)


In [3]:
from pathlib import Path

p = Path(
    r"d:\IITG\Projects\audio_factor_disentanglement_v2\src\models\factorized\content_encoder.py"
)

print(
    "input_channels" in p.read_text()
)

False


In [4]:
# --------------------------------------------------
# Imports
# --------------------------------------------------

import json
import torch
import numpy as np
import pandas as pd

from pathlib import Path

from src.utils.config_loader import load_yaml

In [5]:
# --------------------------------------------------
# Configs
# --------------------------------------------------

model_cfg = load_yaml(
    PROJECT_ROOT
    / "configs"
    / "model_config.yaml"
)

model_cfg.keys()

dict_keys(['factorized_model', 'betas', 'model', 'losses'])

In [6]:
# --------------------------------------------------
# Feature Inventory
# --------------------------------------------------

inventory = pd.read_csv(

    PROJECT_ROOT
    / "data"
    / "metadata"
    / "feature_inventory_v2.csv"
)

print(len(inventory))

inventory.head()

311


,speaker,condition,split,source_file,fragment_id,position_index,vad_region_index,start_sample,end_sample,start_time,...,modgd_shape,logmel_shape,mr_mag_256_shape,mr_phase_256_shape,mr_mag_512_shape,mr_phase_512_shape,mr_mag_1024_shape,mr_phase_1024_shape,available_features,sample_rate
0,s1,clean,train,s1_clean_01.wav,0,0,0,10752,13056,0.672,...,"[513, 37]","[80, 37]","[129, 73]","[129, 73]","[257, 37]","[257, 37]","[513, 19]","[513, 19]","['if', 'logmel', 'magnitude', 'modgd', 'mr_mag...",16000
1,s1,clean,train,s1_clean_01.wav,1,1,0,13056,17408,0.816,...,"[513, 69]","[80, 69]","[129, 137]","[129, 137]","[257, 69]","[257, 69]","[513, 35]","[513, 35]","['if', 'logmel', 'magnitude', 'modgd', 'mr_mag...",16000
2,s1,clean,train,s1_clean_01.wav,2,2,1,18432,21888,1.152,...,"[513, 55]","[80, 55]","[129, 109]","[129, 109]","[257, 55]","[257, 55]","[513, 28]","[513, 28]","['if', 'logmel', 'magnitude', 'modgd', 'mr_mag...",16000
3,s1,clean,train,s1_clean_01.wav,3,3,1,21888,23552,1.368,...,"[513, 27]","[80, 27]","[129, 53]","[129, 53]","[257, 27]","[257, 27]","[513, 14]","[513, 14]","['if', 'logmel', 'magnitude', 'modgd', 'mr_mag...",16000
4,s1,clean,train,s1_clean_01.wav,4,4,1,23552,25088,1.472,...,"[513, 25]","[80, 25]","[129, 49]","[129, 49]","[257, 25]","[257, 25]","[513, 13]","[513, 13]","['if', 'logmel', 'magnitude', 'modgd', 'mr_mag...",16000


In [7]:
clean_row = inventory[

    inventory["condition"] == "clean"

].iloc[3]

clean_row

speaker                                                               s1
condition                                                          clean
split                                                              train
source_file                                              s1_clean_01.wav
fragment_id                                                            3
position_index                                                         3
vad_region_index                                                       1
start_sample                                                       21888
end_sample                                                         23552
start_time                                                         1.368
end_time                                                           1.472
duration                                                           0.104
true_length                                                         1664
padded_length                                      

In [8]:
noisy_row = inventory[

    inventory["condition"] == "noisy"

].iloc[4]

noisy_row

speaker                                                               s1
condition                                                          noisy
split                                                              train
source_file                                              s1_noisy_01.wav
fragment_id                                                            4
position_index                                                         4
vad_region_index                                                       0
start_sample                                                       14720
end_sample                                                         19840
start_time                                                          0.92
end_time                                                            1.24
duration                                                            0.32
true_length                                                         5120
padded_length                                      

In [9]:
def load_fragment_features(row):

    speaker = row["speaker"]

    source_file = Path(
        row["source_file"]
    ).stem

    fragment_id = int(
        row["fragment_id"]
    )

    folder = (

        PROJECT_ROOT
        / "data"
        / "features"
        / speaker
        / source_file
        / f"fragment_{fragment_id:03d}"
    )

    data = {}

    feature_names = [

        "logmel",

        "mr_mag_256",
        "mr_mag_512",
        "mr_mag_1024",

        "magnitude",

        "if",

        "modgd",

        "phase_sin",
        "phase_cos"
    ]

    for name in feature_names:

        data[name] = np.load(
            folder / f"{name}.npy"
        )

    return data

In [10]:
# --------------------------------------------------
# Load Features
# --------------------------------------------------

clean_features = load_fragment_features(
    clean_row
)

noisy_features = load_fragment_features(
    noisy_row
)

print("Loaded Features")

for k,v in clean_features.items():

    print(
        k,
        v.shape
    )

Loaded Features
logmel (80, 27)
mr_mag_256 (129, 53)
mr_mag_512 (257, 27)
mr_mag_1024 (513, 14)
magnitude (513, 27)
if (513, 27)
modgd (513, 27)
phase_sin (513, 27)
phase_cos (513, 27)


In [11]:
# --------------------------------------------------
# Torch Batch
# --------------------------------------------------

batch = {}

for key,value in clean_features.items():

    tensor = torch.tensor(
        value,
        dtype=torch.float32
    )

    tensor = tensor.unsqueeze(0)

    tensor = tensor.unsqueeze(0)

    batch[key] = tensor

for k,v in batch.items():

    print(
        k,
        v.shape
    )

logmel torch.Size([1, 1, 80, 27])
mr_mag_256 torch.Size([1, 1, 129, 53])
mr_mag_512 torch.Size([1, 1, 257, 27])
mr_mag_1024 torch.Size([1, 1, 513, 14])
magnitude torch.Size([1, 1, 513, 27])
if torch.Size([1, 1, 513, 27])
modgd torch.Size([1, 1, 513, 27])
phase_sin torch.Size([1, 1, 513, 27])
phase_cos torch.Size([1, 1, 513, 27])


In [12]:
# --------------------------------------------------
# Imports
# --------------------------------------------------

from src.models.factorized.feature_group_manager import (
    FeatureGroupManager
)

from src.models.factorized.content_encoder import (
    ContentEncoder
)

from src.models.factorized.speaker_encoder import (
    SpeakerEncoder
)

from src.models.factorized.environment_encoder import (
    EnvironmentEncoder
)

from src.models.factorized.excitation_encoder import (
    ExcitationEncoder
)

from src.models.factorized.fidelity_encoder import (
    FidelityEncoder
)

In [13]:
# --------------------------------------------------
# Feature Group Manager
# --------------------------------------------------

group_manager = FeatureGroupManager()

groups = group_manager.build_groups(
    batch
)

groups.keys()

dict_keys(['content', 'speaker', 'environment', 'excitation', 'fidelity'])

In [14]:
# --------------------------------------------------
# Verify Group Structure
# --------------------------------------------------

for group_name,group in groups.items():

    print()

    print(
        "="*60
    )

    print(
        group_name.upper()
    )

    print(
        "="*60
    )

    for feat_name,feat_tensor in group.items():

        print(

            feat_name,

            feat_tensor.shape
        )


CONTENT
logmel torch.Size([1, 1, 80, 27])
mr_mag_256 torch.Size([1, 1, 129, 53])
if torch.Size([1, 1, 513, 27])

SPEAKER
mr_mag_512 torch.Size([1, 1, 257, 27])
mr_mag_256 torch.Size([1, 1, 129, 53])
logmel torch.Size([1, 1, 80, 27])

ENVIRONMENT
magnitude torch.Size([1, 1, 513, 27])
mr_mag_1024 torch.Size([1, 1, 513, 14])
if torch.Size([1, 1, 513, 27])

EXCITATION
modgd torch.Size([1, 1, 513, 27])

FIDELITY
phase_sin torch.Size([1, 1, 513, 27])
phase_cos torch.Size([1, 1, 513, 27])
mr_mag_512 torch.Size([1, 1, 257, 27])
mr_mag_1024 torch.Size([1, 1, 513, 14])
magnitude torch.Size([1, 1, 513, 27])
modgd torch.Size([1, 1, 513, 27])


In [15]:
# --------------------------------------------------
# Encoders
# --------------------------------------------------

content_encoder = ContentEncoder(
    model_cfg
)

speaker_encoder = SpeakerEncoder(
    model_cfg
)

environment_encoder = EnvironmentEncoder(
    model_cfg
)

excitation_encoder = ExcitationEncoder(
    model_cfg
)

fidelity_encoder = FidelityEncoder(
    model_cfg
)

print("Encoders Built")

Encoders Built


In [16]:
# --------------------------------------------------
# Content Encoder
# --------------------------------------------------

content_out = content_encoder(

    groups["content"]
)

print(
    content_out.keys()
)

for k,v in content_out.items():

    print(
        k,
        v.shape
    )

dict_keys(['z', 'mu', 'logvar'])
z torch.Size([1, 64])
mu torch.Size([1, 64])
logvar torch.Size([1, 64])


In [17]:
# --------------------------------------------------
# Speaker Encoder
# --------------------------------------------------

speaker_out = speaker_encoder(

    groups["speaker"]
)

print(
    speaker_out.keys()
)

for k,v in speaker_out.items():

    print(
        k,
        v.shape
    )

dict_keys(['z', 'mu', 'logvar'])
z torch.Size([1, 64])
mu torch.Size([1, 64])
logvar torch.Size([1, 64])


In [18]:
# --------------------------------------------------
# Environment Encoder
# --------------------------------------------------

environment_out = environment_encoder(

    groups["environment"]
)

print(
    environment_out.keys()
)

for k,v in environment_out.items():

    print(
        k,
        v.shape
    )

dict_keys(['z', 'mu', 'logvar'])
z torch.Size([1, 96])
mu torch.Size([1, 96])
logvar torch.Size([1, 96])


In [19]:
# --------------------------------------------------
# Excitation Encoder
# --------------------------------------------------

excitation_out = excitation_encoder(

    groups["excitation"]
)

print(
    excitation_out.keys()
)

for k,v in excitation_out.items():

    print(
        k,
        v.shape
    )

dict_keys(['z', 'mu', 'logvar'])
z torch.Size([1, 32])
mu torch.Size([1, 32])
logvar torch.Size([1, 32])


In [20]:
# --------------------------------------------------
# Fidelity Encoder
# --------------------------------------------------

fidelity_out = fidelity_encoder(

    groups["fidelity"]
)

print(
    fidelity_out.keys()
)

for k,v in fidelity_out.items():

    print(
        k,
        v.shape
    )

dict_keys(['z', 'mu', 'logvar'])
z torch.Size([1, 128])
mu torch.Size([1, 128])
logvar torch.Size([1, 128])


In [21]:
# --------------------------------------------------
# Latent Summary
# --------------------------------------------------

latent_summary = {

    "content":
        content_out["z"],

    "speaker":
        speaker_out["z"],

    "environment":
        environment_out["z"],

    "excitation":
        excitation_out["z"],

    "fidelity":
        fidelity_out["z"]
}

for k,v in latent_summary.items():

    print(
        f"{k:15s}",
        v.shape
    )

content         torch.Size([1, 64])
speaker         torch.Size([1, 64])
environment     torch.Size([1, 96])
excitation      torch.Size([1, 32])
fidelity        torch.Size([1, 128])


In [22]:
# --------------------------------------------------
# Total Latent Capacity
# --------------------------------------------------

total_dim = 0

for k,v in latent_summary.items():

    total_dim += v.shape[-1]

print()

print(
    "Total Latent Dim:",
    total_dim
)


Total Latent Dim: 384


In [23]:
# --------------------------------------------------
# Architecture Verification
# --------------------------------------------------

print()

print(
    "Content      :",
    latent_summary["content"].shape[-1]
)

print(
    "Speaker      :",
    latent_summary["speaker"].shape[-1]
)

print(
    "Environment  :",
    latent_summary["environment"].shape[-1]
)

print(
    "Excitation   :",
    latent_summary["excitation"].shape[-1]
)

print(
    "Fidelity     :",
    latent_summary["fidelity"].shape[-1]
)


Content      : 64
Speaker      : 64
Environment  : 96
Excitation   : 32
Fidelity     : 128


In [24]:
# --------------------------------------------------
# Decoder Core
# --------------------------------------------------

from src.models.factorized.factorized_decoder_core import (
    FactorizedDecoderCore
)

decoder_core = FactorizedDecoderCore(
    model_cfg
)

print(decoder_core.__class__.__name__)

FactorizedDecoderCore


In [25]:
# --------------------------------------------------
# Decoder Core Forward
# --------------------------------------------------

hidden_sequence = decoder_core(

    latent_summary["content"],

    latent_summary["speaker"],

    latent_summary["environment"],

    latent_summary["excitation"],

    latent_summary["fidelity"]
)

print(
    "Hidden Sequence Shape:",
    hidden_sequence.shape
)

Hidden Sequence Shape: torch.Size([1, 32, 256])


In [26]:
# --------------------------------------------------
# Hidden Sequence Statistics
# --------------------------------------------------

print(
    "Min:",
    hidden_sequence.min().item()
)

print(
    "Max:",
    hidden_sequence.max().item()
)

print(
    "Mean:",
    hidden_sequence.mean().item()
)

print(
    "Contains NaN:",
    torch.isnan(hidden_sequence).any().item()
)

print(
    "Contains Inf:",
    torch.isinf(hidden_sequence).any().item()
)

Min: -3.176973581314087
Max: 3.4800615310668945
Mean: -0.019915197044610977
Contains NaN: False
Contains Inf: False


In [27]:
# --------------------------------------------------
# Manual Decoder Core Verification
# --------------------------------------------------

identity_out = decoder_core.identity(

    latent_summary["content"],

    latent_summary["speaker"]
)

print(
    "Identity:",
    identity_out.shape
)

env_out = decoder_core.environment_film(

    identity_out,

    latent_summary["environment"]
)

print(
    "Environment:",
    env_out.shape
)

exc_out = decoder_core.excitation_attention(

    env_out,

    latent_summary["excitation"]
)

print(
    "Excitation:",
    exc_out.shape
)

fid_out = decoder_core.fidelity_attention(

    exc_out,

    latent_summary["fidelity"]
)

print(
    "Fidelity:",
    fid_out.shape
)

Identity: torch.Size([1, 32, 256])
Environment: torch.Size([1, 32, 256])
Excitation: torch.Size([1, 32, 256])
Fidelity: torch.Size([1, 32, 256])


In [28]:
# --------------------------------------------------
# Decoder Heads
# --------------------------------------------------

from src.models.factorized.factorized_decoder import (
    FactorizedDecoder
)

decoder = FactorizedDecoder(
    model_cfg
)

print(
    decoder.__class__.__name__
)

FactorizedDecoder


In [29]:
# --------------------------------------------------
# Decoder Forward
# --------------------------------------------------

reconstructions = decoder(
    hidden_sequence
)

print(
    reconstructions.keys()
)

dict_keys(['logmel', 'mr_mag_256', 'mr_mag_512', 'magnitude', 'mr_mag_1024', 'if', 'modgd', 'phase_sin', 'phase_cos'])


In [30]:
# --------------------------------------------------
# Reconstruction Shapes
# --------------------------------------------------

for name, tensor in reconstructions.items():

    print(
        f"{name:15s}",
        tensor.shape
    )

logmel          torch.Size([1, 80, 32])
mr_mag_256      torch.Size([1, 129, 32])
mr_mag_512      torch.Size([1, 257, 32])
magnitude       torch.Size([1, 513, 32])
mr_mag_1024     torch.Size([1, 513, 32])
if              torch.Size([1, 513, 32])
modgd           torch.Size([1, 513, 32])
phase_sin       torch.Size([1, 513, 32])
phase_cos       torch.Size([1, 513, 32])


In [31]:
# --------------------------------------------------
# Reconstruction Validation
# --------------------------------------------------

for name, tensor in reconstructions.items():

    print(
        f"\n{name}"
    )

    print(
        "Min:",
        tensor.min().item()
    )

    print(
        "Max:",
        tensor.max().item()
    )

    print(
        "NaN:",
        torch.isnan(
            tensor
        ).any().item()
    )

    print(
        "Inf:",
        torch.isinf(
            tensor
        ).any().item()
    )


logmel
Min: 0.3513568937778473
Max: 0.6230303645133972
NaN: False
Inf: False

mr_mag_256
Min: 0.3922058343887329
Max: 0.6209611892700195
NaN: False
Inf: False

mr_mag_512
Min: 0.3937276303768158
Max: 0.6212969422340393
NaN: False
Inf: False

magnitude
Min: 0.3628356456756592
Max: 0.6266596913337708
NaN: False
Inf: False

mr_mag_1024
Min: 0.36598241329193115
Max: 0.6552609801292419
NaN: False
Inf: False

if
Min: -0.5730262994766235
Max: 0.5868823528289795
NaN: False
Inf: False

modgd
Min: -0.6466934680938721
Max: 0.5329747200012207
NaN: False
Inf: False

phase_sin
Min: -0.5496612191200256
Max: 0.5138470530509949
NaN: False
Inf: False

phase_cos
Min: -0.5404826402664185
Max: 0.5488314032554626
NaN: False
Inf: False


In [32]:
# --------------------------------------------------
# Compare Reconstruction vs Target Shapes
# --------------------------------------------------

target_shapes = {

    "logmel":
        batch["logmel"].shape,

    "mr_mag_256":
        batch["mr_mag_256"].shape,

    "mr_mag_512":
        batch["mr_mag_512"].shape,

    "magnitude":
        batch["magnitude"].shape,

    "mr_mag_1024":
        batch["mr_mag_1024"].shape,

    "if":
        batch["if"].shape,

    "modgd":
        batch["modgd"].shape,

    "phase_sin":
        batch["phase_sin"].shape,

    "phase_cos":
        batch["phase_cos"].shape
}

for k in reconstructions:

    print("\n", k)

    print(
        "reconstruction:",
        reconstructions[k].shape
    )

    print(
        "target:",
        target_shapes[k]
    )


 logmel
reconstruction: torch.Size([1, 80, 32])
target: torch.Size([1, 1, 80, 27])

 mr_mag_256
reconstruction: torch.Size([1, 129, 32])
target: torch.Size([1, 1, 129, 53])

 mr_mag_512
reconstruction: torch.Size([1, 257, 32])
target: torch.Size([1, 1, 257, 27])

 magnitude
reconstruction: torch.Size([1, 513, 32])
target: torch.Size([1, 1, 513, 27])

 mr_mag_1024
reconstruction: torch.Size([1, 513, 32])
target: torch.Size([1, 1, 513, 14])

 if
reconstruction: torch.Size([1, 513, 32])
target: torch.Size([1, 1, 513, 27])

 modgd
reconstruction: torch.Size([1, 513, 32])
target: torch.Size([1, 1, 513, 27])

 phase_sin
reconstruction: torch.Size([1, 513, 32])
target: torch.Size([1, 1, 513, 27])

 phase_cos
reconstruction: torch.Size([1, 513, 32])
target: torch.Size([1, 1, 513, 27])


In [33]:
# --------------------------------------------------
# Full Factorized VAE
# --------------------------------------------------

from src.models.factorized.factorized_vae import (
    FactorizedVAE
)

vae = FactorizedVAE(
    model_cfg
)

print(
    vae.__class__.__name__
)

FactorizedVAE


In [34]:
# --------------------------------------------------
# Full Forward Pass
# --------------------------------------------------

outputs = vae(
    batch
)

print(
    outputs.keys()
)

dict_keys(['groups', 'latents', 'mu', 'logvar', 'reconstructions'])


In [35]:
# --------------------------------------------------
# Latent Verification
# --------------------------------------------------

for k,v in outputs[
    "latents"
].items():

    print(
        f"{k:15s}",
        v.shape
    )

content         torch.Size([1, 64])
speaker         torch.Size([1, 64])
environment     torch.Size([1, 96])
excitation      torch.Size([1, 32])
fidelity        torch.Size([1, 128])


In [36]:
# --------------------------------------------------
# Mu Verification
# --------------------------------------------------

for k,v in outputs[
    "mu"
].items():

    print(
        f"{k:15s}",
        v.shape
    )

content         torch.Size([1, 64])
speaker         torch.Size([1, 64])
environment     torch.Size([1, 96])
excitation      torch.Size([1, 32])
fidelity        torch.Size([1, 128])


In [37]:
# --------------------------------------------------
# LogVar Verification
# --------------------------------------------------

for k,v in outputs[
    "logvar"
].items():

    print(
        f"{k:15s}",
        v.shape
    )

content         torch.Size([1, 64])
speaker         torch.Size([1, 64])
environment     torch.Size([1, 96])
excitation      torch.Size([1, 32])
fidelity        torch.Size([1, 128])


In [38]:
# --------------------------------------------------
# Reconstruction Verification
# --------------------------------------------------

for k,v in outputs[
    "reconstructions"
].items():

    print(
        f"{k:15s}",
        v.shape
    )

logmel          torch.Size([1, 80, 32])
mr_mag_256      torch.Size([1, 129, 32])
mr_mag_512      torch.Size([1, 257, 32])
magnitude       torch.Size([1, 513, 32])
mr_mag_1024     torch.Size([1, 513, 32])
if              torch.Size([1, 513, 32])
modgd           torch.Size([1, 513, 32])
phase_sin       torch.Size([1, 513, 32])
phase_cos       torch.Size([1, 513, 32])


In [39]:
# --------------------------------------------------
# Full Forward Safety Check
# --------------------------------------------------

for name, tensor in outputs[
    "reconstructions"
].items():

    print(
        f"{name:15s}",
        "NaN:",
        torch.isnan(
            tensor
        ).any().item()
    )

logmel          NaN: False
mr_mag_256      NaN: False
mr_mag_512      NaN: False
magnitude       NaN: False
mr_mag_1024     NaN: False
if              NaN: False
modgd           NaN: False
phase_sin       NaN: False
phase_cos       NaN: False


In [40]:
# --------------------------------------------------
# Loss Stack
# --------------------------------------------------

from src.losses.total_loss import (
    TotalLoss
)

loss_fn = TotalLoss(
    model_cfg
)

print(
    loss_fn.__class__.__name__
)

TotalLoss


In [41]:
# --------------------------------------------------
# Reconstruction Keys
# --------------------------------------------------

print(

    outputs[
        "reconstructions"
    ].keys()

)

dict_keys(['logmel', 'mr_mag_256', 'mr_mag_512', 'magnitude', 'mr_mag_1024', 'if', 'modgd', 'phase_sin', 'phase_cos'])


In [42]:
# --------------------------------------------------
# Align Targets
# --------------------------------------------------

import torch.nn.functional as F

aligned_targets = {}

for name,tensor in batch.items():

    target = tensor.squeeze(1)

    target = F.interpolate(

        target.unsqueeze(1),

        size=(
            target.shape[1],
            32
        ),

        mode="bilinear",

        align_corners=False

    ).squeeze(1)

    aligned_targets[name] = target

    print(
        name,
        target.shape
    )

logmel torch.Size([1, 80, 32])
mr_mag_256 torch.Size([1, 129, 32])
mr_mag_512 torch.Size([1, 257, 32])
mr_mag_1024 torch.Size([1, 513, 32])
magnitude torch.Size([1, 513, 32])
if torch.Size([1, 513, 32])
modgd torch.Size([1, 513, 32])
phase_sin torch.Size([1, 513, 32])
phase_cos torch.Size([1, 513, 32])


In [43]:
# --------------------------------------------------
# Reconstruction Loss
# --------------------------------------------------

from src.losses.reconstruction_losses import (
    ReconstructionLoss
)

recon_loss = ReconstructionLoss(
    model_cfg
)

result = recon_loss(

    outputs["reconstructions"],

    aligned_targets
)

print(
    result.keys()
)

print(
    "Total Recon Loss:",
    result["loss"]
)

dict_keys(['logmel_loss', 'logmel_l1', 'logmel_mse', 'mr_mag_256_loss', 'mr_mag_256_l1', 'mr_mag_256_mse', 'mr_mag_512_loss', 'mr_mag_512_l1', 'mr_mag_512_mse', 'magnitude_loss', 'magnitude_l1', 'magnitude_mse', 'mr_mag_1024_loss', 'mr_mag_1024_l1', 'mr_mag_1024_mse', 'if_loss', 'if_l1', 'if_mse', 'modgd_loss', 'modgd_l1', 'modgd_mse', 'phase_sin_loss', 'phase_sin_l1', 'phase_sin_mse', 'phase_cos_loss', 'phase_cos_l1', 'phase_cos_mse', 'loss'])
Total Recon Loss: tensor(6.5906, grad_fn=<AddBackward0>)


In [44]:
# --------------------------------------------------
# MR Loss
# --------------------------------------------------

from src.losses.multires_stft_loss import (
    MultiResolutionFeatureLoss
)

mr_loss_fn = (
    MultiResolutionFeatureLoss()
)

mr_loss = mr_loss_fn(

    outputs["reconstructions"],

    aligned_targets

)

print(mr_loss)

tensor(0.8144, grad_fn=<AddBackward0>)


In [45]:
# --------------------------------------------------
# Orthogonality
# --------------------------------------------------

from src.losses.orthogonality_loss import (
    OrthogonalityLoss
)

ortho = OrthogonalityLoss()

ortho_loss = ortho(
    outputs["latents"]
)

print(
    ortho_loss
)

tensor(0.0753, grad_fn=<AddBackward0>)


In [46]:
# --------------------------------------------------
# InfoNCE
# --------------------------------------------------

from src.losses.infonce_loss import (
    InfoNCELoss
)

info = InfoNCELoss(
    model_cfg
)

loss = info(

    outputs["latents"]["content"],

    outputs["latents"]["speaker"]

)

print(loss)

tensor(0., grad_fn=<NllLossBackward0>)


In [47]:
# --------------------------------------------------
# FactorVAE
# --------------------------------------------------

from src.losses.factorvae_loss import (
    FactorVAELoss
)

tc_loss = FactorVAELoss(
    gamma=20
)

dummy_logits = torch.randn(
    8,
    2
)

print(
    tc_loss(
        dummy_logits
    )
)

tensor(3.2853)


In [48]:
loss_fn = TotalLoss(model_cfg)

loss_dict = loss_fn(
    outputs,
    aligned_targets
)

for k,v in loss_dict.items():

    if torch.is_tensor(v):

        print(
            k,
            v.item()
        )

reconstruction 6.590608596801758
logmel_loss 0.23695184290409088
logmel_l1 0.18644554913043976
logmel_mse 0.05050629377365112
mr_mag_256_loss 0.27640193700790405
mr_mag_256_l1 0.21065956354141235
mr_mag_256_mse 0.0657423660159111
mr_mag_512_loss 0.27292484045028687
mr_mag_512_l1 0.20794038474559784
mr_mag_512_mse 0.06498444080352783
magnitude_loss 0.2652907371520996
magnitude_l1 0.20269490778446198
magnitude_mse 0.06259582191705704
mr_mag_1024_loss 0.26508545875549316
mr_mag_1024_l1 0.203001469373703
mr_mag_1024_mse 0.06208398938179016
if_loss 2.2910561561584473
if_l1 0.9893044233322144
if_mse 1.3017516136169434
modgd_loss 1.222151279449463
modgd_l1 0.654267430305481
modgd_mse 0.5678839087486267
phase_sin_loss 0.8663094639778137
phase_sin_l1 0.5096381902694702
phase_sin_mse 0.3566712737083435
phase_cos_loss 0.8944367170333862
phase_cos_l1 0.521778404712677
phase_cos_mse 0.37265828251838684
multires 0.8144122362136841
content_kl 0.0959819108247757
speaker_kl 0.0955670177936554
environme

In [50]:
loss_dict["total"].backward()

print("Backward Success")

Backward Success


In [52]:
from src.models.factorized.factorized_vae import (
    FactorizedVAE
)

model = FactorizedVAE(
    model_cfg
)

print(
    sum(
        p.numel()
        for p in model.parameters()
    )
)

72885349


In [53]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [54]:
optimizer.zero_grad()

outputs = model(batch)

loss_dict = loss_fn(
    outputs,
    aligned_targets
)

loss_dict["total"].backward()

optimizer.step()

print(
    "Loss:",
    loss_dict["total"].item()
)

Loss: 10.935484886169434
